# 05 — Reinforcement Learning: PPO Training
**Goal**: Train the SFT model using Proximal Policy Optimization (PPO) with execution-guided sandbox rewards and KL divergence penalties (KL $\beta=0.02$, LR $1\times 10^{-6}$). Produces `./checkpoints/ppo/final`.

---

## Step 1: Environment & Universal Path Resolution

In [ ]:
!pip install -q "trl<0.12.0" peft transformers datasets

import sys, os, shutil, importlib

# Universal Path Resolution & Auto-Copy for Kaggle / Local / Colab
def prepare_kaggle_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(root, dest)
                print(f"Copied 'src' from {root} to {dest}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(src_dir, dest)
                print(f"Copied 'src' from {src_dir} to {dest}")
                return '/kaggle/working'
    
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src')):
        return parent
    return curr

repo_root = prepare_kaggle_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

# Patch ppo.py in-memory on Kaggle working directory to ensure robust TRL imports, fallback class & Multi-GPU support
ppo_file = os.path.join(repo_root, 'src', 'training', 'ppo.py')
with open(ppo_file, 'w', encoding='utf-8') as f:
    f.write('''"""PPO training entry points with execution-guided rewards."""

import gc
import os
import sys
import torch
import torch.nn as nn

# Robust TRL import ladder with custom PyTorch Fallback ValueHead class
try:
    from trl.models.modeling_value_head import AutoModelForCausalLMWithValueHead
except Exception:
    try:
        from trl import AutoModelForCausalLMWithValueHead
    except Exception:
        try:
            from trl.models import AutoModelForCausalLMWithValueHead
        except Exception:
            AutoModelForCausalLMWithValueHead = None

if AutoModelForCausalLMWithValueHead is None:
    from transformers import AutoModelForCausalLM

    class AutoModelForCausalLMWithValueHead(nn.Module):
        """Fallback Value Head model wrapper if TRL\\\'s class is unavailable in current TRL version."""
        def __init__(self, pretrained_model):
            super().__init__()
            self.pretrained_model = pretrained_model
            self.config = getattr(pretrained_model, "config", None)
            hidden_size = getattr(self.config, "hidden_size", getattr(self.config, "d_model", 2048))
            self.v_head = nn.Linear(hidden_size, 1)
            self.summary_dropout = nn.Dropout(0.1)

        @classmethod
        def from_pretrained(cls, pretrained_model_name_or_path, **kwargs):
            try:
                base_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path, **kwargs)
            except Exception:
                kwargs.pop("attn_implementation", None)
                base_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path, **kwargs)
            return cls(base_model)

        def forward(self, input_ids=None, attention_mask=None, **kwargs):
            outputs = self.pretrained_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, **kwargs)
            last_hidden_state = outputs.hidden_states[-1]
            lm_logits = outputs.logits
            value = self.v_head(self.summary_dropout(last_hidden_state)).squeeze(-1)
            return (lm_logits, None, value)

        def generate(self, *args, **kwargs):
            return self.pretrained_model.generate(*args, **kwargs)

        def save_pretrained(self, save_directory, **kwargs):
            os.makedirs(save_directory, exist_ok=True)
            if hasattr(self.pretrained_model, "save_pretrained"):
                self.pretrained_model.save_pretrained(save_directory, **kwargs)
            torch.save(self.v_head.state_dict(), os.path.join(save_directory, "v_head.bin"))

try:
    from trl.trainer.ppo_trainer import PPOTrainer
except Exception:
    try:
        from trl import PPOTrainer
    except Exception:
        try:
            from trl.trainer import PPOTrainer
        except Exception:
            PPOTrainer = None

try:
    from trl.trainer.ppo_config import PPOConfig
except Exception:
    try:
        from trl import PPOConfig
    except Exception:
        try:
            from trl.trainer import PPOConfig
        except Exception:
            PPOConfig = None

from src.execution.executor import run_code
from src.rewards.execution_reward import compute_reward


def run_ppo_training(
    sft_model_path: str,
    tokenizer,
    dataset,
    output_dir: str = "./checkpoints/ppo",
    num_epochs: int = 1,
    learning_rate: float = 1e-6,
    batch_size: int = 2,
    mini_batch_size: int = 1,
    gradient_accumulation_steps: int = 2,
    init_kl_coef: float = 0.02,
    target_kl: float = 6.0,
    max_steps: int = 10,
):
    print("-> [1/4] Preparing PPO dataset and tokenizer...", flush=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_ppo_prompt(example):
        problem = example.get("question", example.get("prompt", ""))
        prompt_text = f"### Problem:\\n{problem}\\n\\n### Solution:\\n```python\\n"
        tokens = tokenizer(prompt_text, truncation=True, max_length=256)
        return {"input_ids": tokens["input_ids"]}

    if "input_ids" not in dataset.column_names:
        dataset = dataset.map(tokenize_ppo_prompt, remove_columns=dataset.column_names)

    # Clear CUDA memory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"-> [2/4] Available GPUs: {num_gpus}. Loading PPO model with Value Head...", flush=True)

    # Multi-GPU (T4 x2) distribution strategy
    if num_gpus >= 2:
        print("   Multi-GPU detected! Distributing Policy Model across GPU 0 & GPU 1 using device_map=\\\'auto\\\'", flush=True)
        device_map = "auto"
    elif num_gpus == 1:
        print("   Single GPU detected! Using cuda:0 with memory-efficient precision", flush=True)
        device_map = {"": 0}
    else:
        device_map = None

    try:
        ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
            sft_model_path,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map=device_map,
            trust_remote_code=True,
            attn_implementation="eager",
        )
    except Exception:
        ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
            sft_model_path,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map=device_map,
            trust_remote_code=True,
        )

    if hasattr(ppo_model, "config"):
        ppo_model.config.pad_token_id = tokenizer.pad_token_id
        ppo_model.config.use_cache = True
    if hasattr(ppo_model, "generation_config") and ppo_model.generation_config is not None:
        ppo_model.generation_config.pad_token_id = tokenizer.pad_token_id

    # Enable Gradient Checkpointing to save VRAM
    if hasattr(ppo_model, "pretrained_model") and hasattr(ppo_model.pretrained_model, "gradient_checkpointing_enable"):
        try:
            ppo_model.pretrained_model.gradient_checkpointing_enable()
        except Exception:
            pass

    # Freeze base model parameters so only LoRA + Value Head are optimized
    if hasattr(ppo_model, "pretrained_model"):
        for param in ppo_model.pretrained_model.parameters():
            param.requires_grad = False

    for name, param in ppo_model.named_parameters():
        if "lora_" in name or "v_head" in name or "summary" in name or "score" in name:
            param.requires_grad = True

    trainable_params = [p for p in ppo_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate)

    print("-> [3/4] Initializing TRL PPOTrainer...", flush=True)
    ppo_config = PPOConfig(
        model_name=sft_model_path,
        learning_rate=learning_rate,
        batch_size=batch_size,
        mini_batch_size=mini_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        kl_penalty="kl",
        init_kl_coef=init_kl_coef,
        target_kl=target_kl,
    )

    def ppo_collate_fn(data):
        return {key: [d[key] for d in data] for key in data[0]}

    ppo_trainer = PPOTrainer(
        config=ppo_config,
        model=ppo_model,
        ref_model=None,  # TRL manages ref_model using PEFT shared weights
        tokenizer=tokenizer,
        dataset=dataset,
        optimizer=optimizer,
        data_collator=ppo_collate_fn,
    )

    generation_kwargs = {
        "max_new_tokens": 64,
        "do_sample": True,
        "top_p": 0.95,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    step_count = 0
    total_batches = min(len(ppo_trainer.dataloader), max_steps) if max_steps else len(ppo_trainer.dataloader)

    print(f"-> [4/4] Starting PPO Rollout Optimization ({total_batches} steps max)...", flush=True)
    for epoch in range(num_epochs):
        for batch in ppo_trainer.dataloader:
            step_count += 1
            print(f"   [Step {step_count}/{total_batches}] Generating code & executing in sandbox...", flush=True)

            query_tensors = [
                q.squeeze() if isinstance(q, torch.Tensor) and q.dim() > 1 else (torch.tensor(q, dtype=torch.long) if not isinstance(q, torch.Tensor) else q)
                for q in batch["input_ids"]
            ]
            with torch.no_grad():
                response_tensors = ppo_trainer.generate(
                    query_tensors,
                    **generation_kwargs,
                )

            rewards = []
            for q, r in zip(query_tensors, response_tensors):
                code = tokenizer.decode(r, skip_special_tokens=True)
                result = run_code(code, timeout=3)
                reward_val = compute_reward(result["status"], 0, 1)
                rewards.append(torch.tensor(reward_val, dtype=torch.float32))

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
            mean_score = stats.get("ppo/mean_scores", 0.0)
            kl_val = stats.get("objective/kl", 0.0)
            print(f"   ✓ Completed Step {step_count}/{total_batches} | mean_reward={mean_score:.3f} | kl={kl_val:.3f}", flush=True)

            if max_steps and step_count >= max_steps:
                break
        if max_steps and step_count >= max_steps:
            break

    print(f"-> Saving final PPO adapter checkpoint to {output_dir}/final...", flush=True)
    ppo_model.save_pretrained(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    return ppo_trainer
''')
print("SUCCESS: Patched src/training/ppo.py with robust TRL imports & multi-GPU support!")

if 'src.training.ppo' in sys.modules:
    del sys.modules['src.training.ppo']
if 'src.training' in sys.modules:
    del sys.modules['src.training']

import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")


## Step 2: Load APPS Dataset & Tokenizer

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"

# Resolve SFT Checkpoint path on Kaggle input / working
if not os.path.exists(SFT_CHECKPOINT) and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files or 'model.safetensors' in files:
            SFT_CHECKPOINT = root
            break

effective_model = SFT_CHECKPOINT if os.path.exists(SFT_CHECKPOINT) else MODEL_NAME
print(f"Using SFT model checkpoint: {effective_model}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading APPS training dataset for PPO via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:1000]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)
print(f"Prepared {len(apps_clean)} APPS problems for PPO rollout.")

## Step 3: Run Multi-GPU PPO Training with Sandbox Execution Rewards

In [ ]:
import sys, importlib
if 'src.training.ppo' in sys.modules:
    del sys.modules['src.training.ppo']
import src.training.ppo
importlib.reload(src.training.ppo)
from src.training.ppo import run_ppo_training

session_start = time.time()
print("Starting Multi-GPU (T4 x2) PPO Reinforcement Learning Training loop...")

ppo_trainer = run_ppo_training(
    sft_model_path=effective_model,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir="./checkpoints/ppo",
    num_epochs=1,
    learning_rate=1e-6,
    batch_size=2,
    mini_batch_size=1,
    gradient_accumulation_steps=2,
    init_kl_coef=0.02,
    target_kl=6.0,
    max_steps=10,
)

print("\nPPO Training completed successfully!")
print("Saved final PPO adapter checkpoint to ./checkpoints/ppo/final")


## Step 4: Checkpoint Verification & Inference Test
Verifies saved PPO adapter files, reloads weights, confirms LoRA config, and executes 3 APPS inference tests.

In [ ]:
import os
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

checkpoint_dir = None
search_roots = ['./checkpoints/ppo/final', '/kaggle/working/checkpoints/ppo/final', './checkpoints/ppo', '/kaggle/working']

for root in search_roots:
    if os.path.exists(root):
        files_in_root = os.listdir(root) if os.path.isdir(root) else []
        if any(f in files_in_root for f in ['adapter_config.json', 'config.json', 'pytorch_model.bin', 'model.safetensors', 'v_head.bin']):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if any(f in files for f in ['adapter_config.json', 'config.json', 'pytorch_model.bin', 'model.safetensors', 'v_head.bin']) and 'ppo' in r.lower():
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir:
    checkpoint_dir = "./checkpoints/ppo/final"

print(f"=== 1. Inspecting PPO Checkpoint Files in {checkpoint_dir} ===")
if os.path.exists(checkpoint_dir):
    for fname in sorted(os.listdir(checkpoint_dir)):
        fpath = os.path.join(checkpoint_dir, fname)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"  - {fname}: {size_mb:.2f} MB")

    print("\n=== 2. Verifying PPO Model Configuration ===")
    if os.path.exists(os.path.join(checkpoint_dir, 'adapter_config.json')):
        config = PeftConfig.from_pretrained(checkpoint_dir)
        print(f"  - lora_alpha: {getattr(config, 'lora_alpha', 'N/A')}")
        print(f"  - r: {getattr(config, 'r', 'N/A')}")
        print(f"  - target_modules: {list(getattr(config, 'target_modules', []))}")
        print(f"  - peft_type: {getattr(config, 'peft_type', 'N/A')}")
    elif os.path.exists(os.path.join(checkpoint_dir, 'config.json')):
        config = AutoConfig.from_pretrained(checkpoint_dir)
        print(f"  - model_type: {config.model_type}")
        print(f"  - hidden_size: {getattr(config, 'hidden_size', getattr(config, 'd_model', 'N/A'))}")
        print(f"  - vocab_size: {config.vocab_size}")

    print("\n=== 3. Reloading Saved PPO Model ===")
    if os.path.exists(os.path.join(checkpoint_dir, 'adapter_config.json')):
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
        reloaded_ppo = PeftModel.from_pretrained(base_model, checkpoint_dir)
    else:
        reloaded_ppo = AutoModelForCausalLM.from_pretrained(
            checkpoint_dir,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
    reloaded_ppo.eval()
    print("Reloaded PPO model successfully!")

    print("\n=== 4. Running APPS Inference Check (3 Examples) ===")
    sample_prompts = [
        apps_clean[0]['question'],
        apps_clean[1]['question'],
        apps_clean[2]['question']
    ]

    for idx, p in enumerate(sample_prompts):
        prompt_text = f"### Problem:\n{p[:300]}\n\n### Solution:\n```python\n"
        inputs = tokenizer(prompt_text, return_tensors="pt").to(next(reloaded_ppo.parameters()).device)
        with torch.no_grad():
            out = reloaded_ppo.generate(**inputs, max_new_tokens=100, do_sample=False)
        gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"\n--- PPO Inference Sample {idx + 1} ---")
        print(gen_text[:250] + "...")

    print("\nCheckpoint verification completed successfully: checkpoint files, model configuration, model reload, and inference generation verified.")
